# Neuronal Tuning Summary

This notebook has two independent halves:

1. **Cross-session / cross-condition unit-triage aggregator** — runs the per-cluster significance rules over the pre-computed `triage_stats` block stored in every `*_tuning_curves_data.pkl`, joins each cluster with `unit_catalog.csv` for anatomy, and pickles a unit-keyed roll-up so the same physical unit recorded across replicate sessions in a day is represented once with per-session evidence stacked underneath each modality.

2. **Anatomy / dataset-overview figures** — renders the per-session SVG / PNG anatomy panels that summarise the recording corpus (recording yield by mouse and by cell type, per-probe unit waveforms with the four-shank schematic, and a 360° rotating brain video showing every SU-somatic unit's 3D position coloured by brain-area bucket). These read straight from `unit_catalog.csv` and the per-session Kilosort outputs; they do NOT depend on the aggregator pickle.

The triage stats themselves are produced by `generate-rm` — this notebook is a pure pkl-to-pickle / catalog-to-figure pass and never touches spike or USV data. Thresholds default to `_parameter_settings/analyses_settings.json` (`detect_interesting_tuning_neurons` block) and can be overridden in the configuration cell below.


In [ ]:
### Imports

import json
from pathlib import Path

from usv_playpen.analyses.unit_triage_aggregator import aggregate_units_across_conditions
from usv_playpen.visualizations.make_anatomy_figures import AnatomyFigureMaker


## Configure thresholds

Thresholds default to the values in `_parameter_settings/analyses_settings.json` (`detect_interesting_tuning_neurons` block); override here if you want to sweep different values across the same set of pkls without re-computing tuning.

In [ ]:
THRESHOLDS = {
    "z_threshold":               3.0,
    "min_consecutive_bins":        3,
    "vmi_alpha":                0.01,
    "vmi_min_bouts":              10,
    "spatial_info_bps_threshold": 0.5,
}


## Cross-session / cross-condition aggregator

Collapses same-day duplicate units across the sessions in `CONDITION_TO_SESSION_LIST` into a single pickle keyed by `(animal_id, YYYYMMDD, imec, cluster_id)`. Each unit carries its identity, the catalog `anatomy_region`, and a `conditions` block — one entry per condition listing, per modality, `n_significant`, `n_tested`, `consistency`, an aggregate scalar, and per-session evidence rows.

Each value of `CONDITION_TO_SESSION_LIST` is a path to a `.txt` file with one session root per line (the `ephys_courtship_*_sessions_list.txt` lists under `/mnt/falkner/Bartul/modeling/input_files/`). Sessions missing a `tuning_curves` directory are recorded under `sessions_skipped` and not counted. Anatomy is read from the authoritative `unit_catalog.csv`; orphan pkls (no catalog row) raise.

The output is written to `<out_dir>/unit_triage_<YYYYMMDD>_<HHMMSS>.pkl` and is the input to all downstream cross-session plotting.

In [ ]:
CONDITION_TO_SESSION_LIST = {
    "intact_female": "/mnt/falkner/Bartul/modeling/input_files/ephys_courtship_intact_partners_sessions_list.txt",
    "mute_female":   "/mnt/falkner/Bartul/modeling/input_files/ephys_courtship_mute_female_sessions_list.txt",
}

CATALOG_PATH = "/mnt/falkner/Bartul/EPHYS/unit_catalog.csv"
AGGREGATOR_OUT_DIR = "/mnt/falkner/Bartul/neuronal_tuning"
DATA_ROOT = "/mnt/falkner/Bartul/Data"


In [ ]:
aggregator_pkl_path = aggregate_units_across_conditions(
    condition_to_session_list=CONDITION_TO_SESSION_LIST,
    catalog_path=CATALOG_PATH,
    out_dir=AGGREGATOR_OUT_DIR,
    data_root=DATA_ROOT,
    z_threshold=THRESHOLDS["z_threshold"],
    min_consecutive_bins=THRESHOLDS["min_consecutive_bins"],
    vmi_alpha=THRESHOLDS["vmi_alpha"],
    vmi_min_bouts=THRESHOLDS["vmi_min_bouts"],
    spatial_info_bps_threshold=THRESHOLDS["spatial_info_bps_threshold"],
    message_output=print,
)

print(f"\nAggregator wrote: {aggregator_pkl_path}")


## Anatomy / dataset-overview figures

Three corpus-level figures rendered straight from `unit_catalog.csv` and the per-session Kilosort outputs. Each method on `AnatomyFigureMaker` writes one timestamped file. Output directory, format, and dpi default to the `figures` block of `visualizations_settings.json`; pass `out_dir=...` to override per call.

* **Recording yield** — two-panel stacked bar of SU-somatic / SU-nonsomatic / MUA counts (per-mouse on the left, per-cell-type split by anatomy bucket on the right). Cross-session summary; no session arg needed.

* **Per-probe unit waveforms** — for one chosen `(mouse_id, session_id)` pair, the mean Kilosort templates of the top-amplitude SU-somatic clusters are drawn on a four-shank schematic, one figure per probe. The schematic sits on the right for `imec0` (RH) and on the left for `imec1` (LH); shank ordering is auto-flipped so rostral sits on the left in both probes.

* **Unit positions video** — every SU-somatic unit in the corpus rendered as a 3D dot inside the Allen CCF brain mesh, coloured by its anatomy bucket. The camera sweeps a full 360° in `n_frames` frames at `fps`, producing an mp4 / gif.

In [ ]:
### Build the AnatomyFigureMaker

with open(Path.cwd().parent / "_parameter_settings" / "visualizations_settings.json") as f:
    vis_settings = json.load(f)

anatomy_maker = AnatomyFigureMaker(
    catalog_path=CATALOG_PATH,
    visualizations_parameter_dict=vis_settings,
    message_output=print,
)


In [ ]:
### Recording-yield figure (per-mouse + per-cell-type stacked bars)

yield_path = anatomy_maker.make_recording_yield_figure()
print(f"wrote {yield_path}")


In [ ]:
### Per-probe unit waveforms

# One `(mouse_id, session_id)` pair → two figures (one per probe).
# Schematic side flips between probes so rostral is on the left for
# both. Override `(mouse_id, session_id)` here to render a different
# session.
ANATOMY_WAVEFORM_MOUSE = "158114_2"
ANATOMY_WAVEFORM_SESSION = "20241115_162223"

for probe, schematic_side in (("imec0", "right"), ("imec1", "left")):
    wf_path = anatomy_maker.make_unit_waveform_figure(
        mouse_id=ANATOMY_WAVEFORM_MOUSE,
        session_id=ANATOMY_WAVEFORM_SESSION,
        probe_filter=probe,
        schematic_side=schematic_side,
    )
    print(f"wrote {wf_path}")


In [ ]:
### 360° rotating brain video

# `n_frames=180` at `fps=30` produces a 6-second video. Bump fps for
# smoother playback; bump n_frames to slow the rotation.
video_path = anatomy_maker.make_unit_positions_video(
    n_frames=180,
    fps=30,
    video_format="mp4",
)
print(f"wrote {video_path}")
